In [19]:
##### SET UP & VARIABLES ###########################################################################

import pandas as pd
import geopandas as gpd
import json
import numpy as np
import mapclassify

# --------------------------------------------------
# ADJUSTABLE: Folder path and file names
# --------------------------------------------------
data_folder = 'CTs/'
neighborhoods = ['High', 'Low', 'Home']
file_suffix_2013 = '_CT13_json.geojson'
file_suffix_2021 = '_CT21_json.geojson'

# --------------------------------------------------
# ADJUSTABLE: Exact Column Names from your data
# --------------------------------------------------
merge_key = 'Census_Tract'
col_income = 'med_incomeE'
col_property = 'MEDIAN_AssessedConverted'

# Dictionary mapping exact estimate columns to display names
demo_cols = {
    'white_aloneE': 'White',
    'black_aloneE': 'Black',
    'AIAN_aloneE': 'Indigenous (AIAN)',
    'asian_aloneE': 'Asian',
    'NHPI_aloneE': 'Pacific Islander (NHPI)',
    'some_other_raceE': 'Other Race',
    'two_or_more_racesE': 'Two or More Races',
    'hisp_latinoE': 'Latino or Hispanic'
}

all_processed_gdfs = []

In [20]:
##### THE PROCESSING LOOP ###########################################################################

for neigh in neighborhoods:
    # Load 2013 and 2021 files for the current neighborhood from the CTs folder
    file_2013 = f"{data_folder}{neigh}{file_suffix_2013}"
    file_2021 = f"{data_folder}{neigh}{file_suffix_2021}"
    
    try:
        gdf_13 = gpd.read_file(file_2013)
        gdf_21 = gpd.read_file(file_2021)
    except Exception as e:
        print(f"Error loading files for {neigh}: {e}")
        continue
        
    # ----- TWO LINES TO FIX MISMATCH OF TRACTS -----
    # Force both columns to be formatted as text strings and strip any weird hidden spaces
    gdf_13[merge_key] = gdf_13[merge_key].astype(str).str.strip()
    gdf_21[merge_key] = gdf_21[merge_key].astype(str).str.strip()

    # Strip any ".0" decimals that Pandas might have added to the numbers
    gdf_13[merge_key] = gdf_13[merge_key].str.replace(r'\.0$', '', regex=True)
    gdf_21[merge_key] = gdf_21[merge_key].str.replace(r'\.0$', '', regex=True)
    
    # Set index to Census_Tract for easy subtraction and merging
    gdf_13.set_index(merge_key, inplace=True)
    gdf_21.set_index(merge_key, inplace=True)
    
    # Base our new output on the 2021 geometries
    gdf_change = gdf_21[['geometry', 'NeighID']].copy()
    
    # 1. & 2. Income and Property Value raw change
    gdf_change['income_change'] = gdf_21[col_income] - gdf_13[col_income]
    gdf_change['property_value_change'] = gdf_21[col_property] - gdf_13[col_property]
    
    # 3. Demographic Percent Changes
    top_3_list = []
    
    for tract in gdf_change.index:
        if tract in gdf_13.index and tract in gdf_21.index:
            row_13 = gdf_13.loc[tract]
            row_21 = gdf_21.loc[tract]
            
            changes = []
            for est_col, display_name in demo_cols.items():
                moe_col = est_col[:-1] + 'M' # Replace 'E' with 'M' to get MOE column
                
                val_13 = row_13[est_col] if pd.notnull(row_13[est_col]) else 0
                val_21 = row_21[est_col] if pd.notnull(row_21[est_col]) else 0
                
                moe_13 = row_13[moe_col] if pd.notnull(row_13[moe_col]) else 0
                moe_21 = row_21[moe_col] if pd.notnull(row_21[moe_col]) else 0
                
                # Calculate percent change safely
                if val_13 > 0:
                    pct_change = ((val_21 - val_13) / val_13) * 100
                elif val_13 == 0 and val_21 > 0:
                    pct_change = 100.0 # Cap infinite growth at 100%
                else:
                    pct_change = 0.0
                
                changes.append({
                    'group': display_name,
                    'pct_change': round(pct_change, 1),
                    'est_13': int(val_13),
                    'moe_13': int(moe_13),
                    'est_21': int(val_21),
                    'moe_21': int(moe_21)
                })
                
            # Sort by absolute percent change to find the biggest movers
            sorted_changes = sorted(changes, key=lambda x: abs(x['pct_change']), reverse=True)
            top_3 = sorted_changes[:3]
            top_3_list.append(json.dumps(top_3))
        else:
            top_3_list.append(json.dumps([]))
            
    gdf_change['top_3_demographics'] = top_3_list
    
    # Reset index to bring Census_Tract back as a normal column
    gdf_change.reset_index(inplace=True)
    all_processed_gdfs.append(gdf_change)
    print(f"Processed {neigh} successfully.")

Processed High successfully.
Processed Low successfully.
Processed Home successfully.


In [21]:
##### COMBINE 3 N-HOODS, CALCULATE BINS, CREATE TABLE, & EXPORT ############################################

# Combine all 3 neighborhoods into one master GeoDataFrame
if all_processed_gdfs:
    final_gdf = pd.concat(all_processed_gdfs, ignore_index=True)
    
    # =====================================================================
    # PART A: CALCULATE GLOBAL BINS FOR THE MAP (For consistent legends)
    # =====================================================================
    import mapclassify

    # 1. Calculate global Natural Breaks (5 for Income, 3 for Property) using final_gdf
    income_breaks = mapclassify.NaturalBreaks(final_gdf['income_change'].dropna(), k=5).bins
    prop_breaks = mapclassify.NaturalBreaks(final_gdf['property_value_change'].dropna(), k=3).bins

    # 2. Print these out! (We will need to look at these for the JavaScript step later)
    print("--- COPY THESE DOLLAR AMOUNTS LATER FOR YOUR JS LEGEND ---")
    print(f"Income Breaks: {income_breaks}")
    print(f"Property Breaks: {prop_breaks}")
    print("----------------------------------------------------------")

    # 3. Functions to assign the bin numbers based on the global breaks
    def assign_income_bin(val):
        if pd.isna(val): return None
        if val <= income_breaks[0]: return 1
        if val <= income_breaks[1]: return 2
        if val <= income_breaks[2]: return 3
        if val <= income_breaks[3]: return 4
        return 5

    def assign_prop_bin(val):
        if pd.isna(val): return None
        if val <= prop_breaks[0]: return 1
        if val <= prop_breaks[1]: return 2
        return 3

    # 4. Save these universal bin numbers directly into the map data
    final_gdf['income_bin_global'] = final_gdf['income_change'].apply(assign_income_bin)
    final_gdf['prop_bin_global'] = final_gdf['property_value_change'].apply(assign_prop_bin)

    # 5. --- NEW: DISSOLVE THE TRACTS TO REMOVE INNER BORDERS ---
    # This acts like a cookie cutter, merging all pieces of the same tract together
    final_gdf_clean = final_gdf.dissolve(by='Census_Tract', as_index=False)

    # 6. Export your main map file (Overwrite your old one using the CLEAN dataframe)
    final_gdf_clean.to_file("combined_neighborhoods_processed.geojson", driver="GeoJSON")


    # =====================================================================
    # PART B: PREPARE THE LIGHTWEIGHT TABLE DATA (For Page 2)
    # =====================================================================

    # 1. Load the 6 raw tract files 
    high_21 = gpd.read_file("CTs/High_CT21_json.geojson")
    low_21  = gpd.read_file("CTs/Low_CT21_json.geojson")
    home_21 = gpd.read_file("CTs/Home_CT21_json.geojson")

    high_13 = gpd.read_file("CTs/High_CT13_json.geojson")
    low_13  = gpd.read_file("CTs/Low_CT13_json.geojson")
    home_13 = gpd.read_file("CTs/Home_CT13_json.geojson")

    # 2. Add the custom "ACS_period" column
    high_21['ACS_period'] = "2017-2021"
    low_21['ACS_period']  = "2017-2021"
    home_21['ACS_period'] = "2017-2021"

    high_13['ACS_period'] = "2009-2013"
    low_13['ACS_period']  = "2009-2013"
    home_13['ACS_period'] = "2009-2013"

    # 3. Combine them all into one dataframe
    all_tracts = pd.concat([high_21, low_21, home_21, high_13, low_13, home_13], ignore_index=True)

    # 4. Keep ONLY the specific columns you requested to keep the file small and fast
    columns_to_keep = [
        'NeighID', 'Census_Tract', 'ACS_period', 'med_incomeE', 'MEDIAN_AssessedConverted', 
        'total_estimateE', 'white_aloneE', 'black_aloneE', 'AIAN_aloneE', 
        'asian_aloneE', 'NHPI_aloneE', 'some_other_raceE', 'two_or_more_racesE', 'hisp_latinoE'
    ]
    table_df = all_tracts[columns_to_keep]

    # 5. Sort the data: Neighborhood (A-Z) -> Census Tract (Low to High) -> Year (2021 then 2013)
    table_df = table_df.sort_values(by=['NeighID', 'Census_Tract', 'ACS_period'], ascending=[True, True, False])

    # 6. Export to a tiny JSON file for the webpage
    table_df.to_json("table_data.json", orient="records")
    
    print("Success! Data processed, global bins calculated, tracts dissolved, and both files saved.")

else:
    print("No data processed. Check file paths and names.")

--- COPY THESE DOLLAR AMOUNTS LATER FOR YOUR JS LEGEND ---
Income Breaks: [18183. 21119. 26495. 30029. 50177.]
Property Breaks: [ 37690.  98960. 209310.]
----------------------------------------------------------
Success! Data processed, global bins calculated, tracts dissolved, and both files saved.
